In [ ]:
library(Signac)
library(Seurat)
library(dplyr)
library(sceasy)
library(reticulate)
use_condaenv('multi_integration')
library(future)

plan("multicore", workers = 8)
options(future.globals.maxSize = 100000 * 1024^3)

set.seed(1234)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays: Please run: options(Seurat.object.assay.version = 'v5')


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: reticulate



### Load RNA data

In [ ]:
rna <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
rna
format(object.size(rna), units = "Mb")
format(object.size(rna), units = "Gb")

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

[1] "50529.1 Mb"

[1] "49.3 Gb"

In [ ]:
# Check condition and update values
rna$Batch_for_correction[rna$SampleID == 'GNG_region_11'] <- '10Xv1_nuclei_multiome'

In [ ]:
rna <- AddMetaData(rna, readRDS('/projects/0/einf2548/cruiz/dmg/data/iCNV_dmg_atlas.rds')) 

rna$cell_id <- paste0(rna$predicted.annotation_level_3, '_', rna$iCNV)

In [ ]:
rna$cell_id <- dplyr::recode(rna$cell_id,
                  `NPC-like_normal` = 'Stem_like_normal', 
                   `OPC_normal` = 'Stem_like_normal', 
                   `OPC-like_normal` = 'Stem_like_normal', 
                   `Endothelial_normal` = 'Endothelial', 
                   `TAM-BDM_normal` = 'TAM_BDM', 
                   `Mono_normal` = 'Mono', 
                   `Oligodendrocyte_normal` = 'Oligodendrocyte', 
                   `CD4/CD8_normal` = 'CD4_CD8', 
                   `MES-like_normal` = 'Differentiated_like_normal', 
                   `TAM-MG_normal` = 'TAM_MG', 
                   `B cell_normal` = 'B_cell', 
                   `DC_normal` = 'DC', 
                   `Mural cell_normal` = "Mural_cell", 
                   `Endothelial_tumor` = 'Endothelial', 
                   `TAM-BDM_tumor` = 'TAM_BDM', 
                   `Mono_tumor` = 'Mono', 
                   `Plasma B_normal` = 'Plasma', 
                   `Oligodendrocyte_tumor` = 'Oligodendrocyte', 
                   `Mural cell_tumor` = 'Mural_cell', 
                   `Plasma B_tumor` = 'Plasma', 
                   `MES-like_tumor`  = 'Differentiated_like_tumor', 
                   `CD4/CD8_tumor` = 'CD4_CD8', 
                   `OPC-like_tumor` = 'Stem_like_tumor', 
                   `AC-like_tumor` = 'Differentiated_like_tumor', 
                   `NPC-like_tumor` = 'Stem_like_tumor', 
                   `AC-like_normal` = 'Differentiated_like_normal', 
                   `TAM-MG_tumor` = 'TAM_MG', 
                   `B cell_tumor` = 'B_cell', 
                   `OPC_tumor` = 'Stem_like_tumor', 
                   `Neuron_normal` = 'Neuron', 
                   `Mast_tumor` = 'Mast', 
                   `Astrocyte_tumor` = 'Differentiated_like_tumor', 
                   `Neuron_tumor` = 'Neuron', 
                   `Astrocyte_normal` = 'Differentiated_like_normal', 
                   `DC_tumor` = 'DC', 
                   `Mast_normal` = 'Mast',
                   `NK_normal` = 'CD4_CD8')

In [ ]:
rna_only <- subset(rna, Batch_for_correction != "10Xv1_nuclei_multiome")
rna_multi <- subset(rna, Batch_for_correction == "10Xv1_nuclei_multiome")
rna_only
rna_multi

An object of class Seurat 
38576 features across 322949 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

An object of class Seurat 
38576 features across 86612 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
rm(rna)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,9471815,505.9,16520452,882.3,16520452,882.3
Vcells,13017269789,99313.9,16789084404,128090.6,13266198152,101213.1


### Load ATAC data

In [ ]:
atac_only <- readRDS('data/atac_common_peak_set_dmg_atlas_qc_filtered_dbl_scores_merged.rds')
atac_only
atac_only <- subset(atac_only, combined > 0.05)
atac_only

An object of class Seurat 
378298 features across 43608 samples within 1 assay 
Active assay: ATAC (378298 features, 0 variable features)
 2 layers present: counts, data

In [ ]:
atac_multi <- readRDS('data/multiome_common_peak_set_dmg_atlas_qc_filtered_dbl_scores_merged.rds')
atac_multi
atac_multi <- subset(atac_multi, combined > 0.05)
atac_multi

An object of class Seurat 
378298 features across 99736 samples within 1 assay 
Active assay: ATAC (378298 features, 0 variable features)
 2 layers present: counts, data

In [ ]:
head(rownames(atac_only))

[1] "chr1-9729-10741"    "chr1-15764-16590"   "chr1-17055-17946"  
[4] "chr1-28900-29821"   "chr1-180694-181903" "chr1-183749-184822"

#### Add metadata

##### ATAC only

In [ ]:
meta_cohort <- readxl::read_excel('../../data/dmg_atlas_metadata.xlsx',
                                  sheet = "Metadata")
meta_cohort

Study,Institute,Preservation_method,ID,Diagnosis,Tumor_type,Tumor_subtype,Location,Source,Clinical_status,⋯,PPM1D,ACVR1,TSHR,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Ruiz2023,PMC,Cryo,T18-90532,DMG Pons,DMG H3 K27-altered,H3.1 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T19-90171,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T19-91014,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-01237,DMG Spinal,DMG H3 K27-altered,H3.3 K27-mutant,Spinal,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-90151,DMG Thalamus,DMG H3 K27-altered,H3.3 K27-mutant,Thalamus,Biopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-90239,DMG Spinal,DMG H3 K27-altered,H3.3 K27-mutant,Spinal,Biopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-90296,DMG Pons,DMG H3 K27-altered,H3.2 K27-mutant,Pons,Biopsy,Primary/Recurrence,⋯,WT,M,WT,WT,WT,WT,WT,WT,WT,FGFR3 overexpression; TP53 inactivation
Ruiz2023,PMC,Cryo,T20-93623,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,RGMA-ERAS fusion
Ruiz2023,PMC,Cryo/Snap-frozen,T20-93210,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A


In [ ]:
integrated <- readRDS('data/cca_integrated_atac_multiome_dmg_atlas_no_doublets.rds')

In [ ]:
# Assuming you have a dataframe named "data" containing the column you want to rename
colnames(atac_only@meta.data) <- sub("p.value", "amulet_pvalue", colnames(atac_only@meta.data))
colnames(atac_only@meta.data) <- sub("q.value", "amulet_qvalue", colnames(atac_only@meta.data))
colnames(atac_only@meta.data) <- sub("combined", "dbl_combined_score", colnames(atac_only@meta.data))

head(atac_only@meta.data)

,orig.ident,total,duplicate,chimeric,unmapped,lowmapq,mitochondrial,nonprimary,passed_filters,is__cell_barcode,⋯,pct_reads_in_peaks,blacklist_ratio,high.tss,nucleosome_group,scDblFinder.score,scDblFinder.class,amulet_pvalue,amulet_qvalue,scDblFinder.p,dbl_combined_score
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
atac_T19-91014_AAACGAAAGACCCTAT-1,SeuratProject,37892,15065,1,438,3127,45,5,19211,1,⋯,73.68174,0.005604587,High,NS < 4,0.002426961,singlet,7.109960e-01,1.000000e+00,0.99757304,0.95291788
atac_T19-91014_AAACGAAAGCGTTGCC-1,SeuratProject,22601,6146,0,246,1761,1,3,14444,1,⋯,65.59817,0.007014953,High,NS < 4,0.001284394,singlet,9.999999e-01,1.000000e+00,0.99871561,0.99999917
atac_T19-91014_AAACGAAAGTACAACA-1,SeuratProject,88086,16394,15,1025,10885,82,198,59487,1,⋯,26.92857,0.009146758,High,NS < 4,0.982575774,doublet,5.340977e-01,1.000000e+00,0.01742423,0.05283217
atac_T19-91014_AAACGAAAGTAGTGTA-1,SeuratProject,67957,14538,11,877,7296,22,43,45170,1,⋯,43.59088,0.006111504,High,NS < 4,0.328079313,singlet,3.778549e-06,3.313147e-05,0.67192069,0.00558055
atac_T19-91014_AAACGAAAGTGAATAC-1,SeuratProject,21580,5723,1,213,2597,9,12,13025,1,⋯,63.17083,0.006865955,High,NS < 4,0.001355713,singlet,9.999999e-01,1.000000e+00,0.99864429,0.99999908
atac_T19-91014_AAACGAAAGTGCCCTG-1,SeuratProject,28527,7851,0,319,2382,16,6,17953,1,⋯,42.65582,0.005032502,High,NS < 4,0.015990626,singlet,9.999927e-01,1.000000e+00,0.98400937,0.99987135


In [ ]:
table(atac_only$SampleID)


         MUV1         MUV16         MUV35         MUV86 P-1764_S-1766 
         6654          2645          6893          5506          1000 
P-2687_S-2688 P-6253_S-8498 P-6255_S-8500      SUDIPG55     T19-91014 
          461           709           257          1163          8742 
    T20-90296 
         9578 

In [ ]:
atac_only$ID <- atac_only$SampleID
meta_id_cells <- data.frame(Cell = colnames(atac_only),
           ID = atac_only$ID,
           check.names = FALSE
          )
meta_id_cells

,Cell,ID
,<chr>,<chr>
atac_T19-91014_AAACGAAAGACCCTAT-1,atac_T19-91014_AAACGAAAGACCCTAT-1,T19-91014
atac_T19-91014_AAACGAAAGCGTTGCC-1,atac_T19-91014_AAACGAAAGCGTTGCC-1,T19-91014
atac_T19-91014_AAACGAAAGTACAACA-1,atac_T19-91014_AAACGAAAGTACAACA-1,T19-91014
atac_T19-91014_AAACGAAAGTAGTGTA-1,atac_T19-91014_AAACGAAAGTAGTGTA-1,T19-91014
atac_T19-91014_AAACGAAAGTGAATAC-1,atac_T19-91014_AAACGAAAGTGAATAC-1,T19-91014
atac_T19-91014_AAACGAAAGTGCCCTG-1,atac_T19-91014_AAACGAAAGTGCCCTG-1,T19-91014
atac_T19-91014_AAACGAAAGTGTAATG-1,atac_T19-91014_AAACGAAAGTGTAATG-1,T19-91014
atac_T19-91014_AAACGAACAAATGCTC-1,atac_T19-91014_AAACGAACAAATGCTC-1,T19-91014
atac_T19-91014_AAACGAACACGCCGAT-1,atac_T19-91014_AAACGAACACGCCGAT-1,T19-91014


In [ ]:
all(meta_id_cells$ID %in% meta_cohort$ID)

[1] TRUE

In [ ]:
atac_only_meta <- list(meta_id_cells, meta_cohort) %>% 
                    purrr::reduce(inner_join, by = 'ID') %>% 
                    distinct(Cell, .keep_all = TRUE) %>% 
                    tibble::column_to_rownames('Cell')
atac_only_meta

,ID,Study,Institute,Preservation_method,Diagnosis,Tumor_type,Tumor_subtype,Location,Source,Clinical_status,⋯,PPM1D,ACVR1,TSHR,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
atac_T19-91014_AAACGAAAGACCCTAT-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGCGTTGCC-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGTACAACA-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGTAGTGTA-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGTGAATAC-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGTGCCCTG-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAAAGTGTAATG-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAACAAATGCTC-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
atac_T19-91014_AAACGAACACGCCGAT-1,T19-91014,Ruiz2023,PMC,Cryo,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A


In [ ]:
atac_only <- AddMetaData(atac_only, atac_only_meta)
atac_only <- AddMetaData(atac_only, FetchData(subset(integrated, 
                                                     dataset == 'ATAC'), 
                                              'predicted.id')
                         )
atac_only$orig.ident <- NULL
atac_only$Batch_for_correction <- recode(atac_only$Sc_platform_ATAC,
                                        `10x_v1` = '10Xv1_nuclei_atac',
                                        `10X_v1.1` = '10X_v1.1_nuceli_atac')
atac_only$cell_id <- atac_only$predicted.id
atac_only@meta.data

,total,duplicate,chimeric,unmapped,lowmapq,mitochondrial,nonprimary,passed_filters,is__cell_barcode,excluded_reason,⋯,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other,predicted.id,Batch_for_correction,cell_id
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
atac_T19-91014_AAACGAAAGACCCTAT-1,37892,15065,1,438,3127,45,5,19211,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,10x_v1.1,Stem_like_tumor
atac_T19-91014_AAACGAAAGCGTTGCC-1,22601,6146,0,246,1761,1,3,14444,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,TAM_MG,10x_v1.1,TAM_MG
atac_T19-91014_AAACGAAAGTACAACA-1,88086,16394,15,1025,10885,82,198,59487,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,10x_v1.1,Stem_like_tumor
atac_T19-91014_AAACGAAAGTAGTGTA-1,67957,14538,11,877,7296,22,43,45170,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,10x_v1.1,Stem_like_tumor
atac_T19-91014_AAACGAAAGTGAATAC-1,21580,5723,1,213,2597,9,12,13025,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Oligodendrocyte,10x_v1.1,Oligodendrocyte
atac_T19-91014_AAACGAAAGTGCCCTG-1,28527,7851,0,319,2382,16,6,17953,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,TAM_MG,10x_v1.1,TAM_MG
atac_T19-91014_AAACGAAAGTGTAATG-1,27213,9888,2,323,2030,71,9,14890,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,10x_v1.1,Stem_like_tumor
atac_T19-91014_AAACGAACAAATGCTC-1,32533,8763,0,352,2536,206,14,20662,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Differentiated_like_tumor,10x_v1.1,Differentiated_like_tumor
atac_T19-91014_AAACGAACACGCCGAT-1,17835,5161,3,213,1943,74,16,10425,1,0,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,10x_v1.1,Stem_like_tumor


##### Multiome

In [ ]:
# Assuming you have a dataframe named "data" containing the column you want to rename
colnames(atac_multi@meta.data) <- sub("p.value", "amulet_pvalue", colnames(atac_multi@meta.data))
colnames(atac_multi@meta.data) <- sub("q.value", "amulet_qvalue", colnames(atac_multi@meta.data))
colnames(atac_multi@meta.data) <- sub("combined", "dbl_combined_score", colnames(atac_multi@meta.data))

head(atac_multi@meta.data)

,orig.ident,gex_barcode,atac_barcode,is_cell,excluded_reason,gex_raw_reads,gex_mapped_reads,gex_conf_intergenic_reads,gex_conf_exonic_reads,gex_conf_intronic_reads,⋯,pct_reads_in_peaks,blacklist_ratio,high.tss,nucleosome_group,scDblFinder.score,scDblFinder.class,amulet_pvalue,amulet_qvalue,scDblFinder.p,dbl_combined_score
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
GNG_region_11_AAACAGCCAAAGCGGC-1,SeuratProject,AAACAGCCAAAGCGGC-1,ACAGCGGGTGGATTAG-1,0,2,4828,4639,410,1182,2776,⋯,13.180516,0.011647727,Low,NS < 4,0.020549510,singlet,1.000000e+00,1.000000e+00,0.9794505,0.999787397
GNG_region_11_AAACAGCCAGGTTCAC-1,SeuratProject,AAACAGCCAGGTTCAC-1,ACAGCGGGTCCTGACA-1,1,0,11573,10964,1090,1980,5696,⋯,24.829869,0.004587156,High,NS < 4,0.120021418,singlet,1.000000e+00,1.000000e+00,0.8799786,0.992490629
GNG_region_11_AAACATGCAACTGGCT-1,SeuratProject,AAACATGCAACTGGCT-1,CATTTAGGTCTTGACG-1,1,0,3339,3187,234,648,2149,⋯,16.735623,0.001241465,High,NS < 4,0.001335426,singlet,1.000000e+00,1.000000e+00,0.9986646,0.999999108
GNG_region_11_AAACATGCACGAACAG-1,SeuratProject,AAACATGCACGAACAG-1,CATTTAGGTCACACTG-1,1,0,15407,14885,969,6106,7092,⋯,18.917511,0.011389680,High,NS < 4,0.202490866,singlet,9.999859e-01,1.000000e+00,0.7975091,0.977952592
GNG_region_11_AAACATGCAGGCCAAA-1,SeuratProject,AAACATGCAGGCCAAA-1,CATTTAGGTTGAGGTA-1,0,2,127096,122482,9153,26593,80542,⋯,7.329515,0.018049883,High,NS < 4,0.037750602,singlet,4.135571e-56,9.657695e-55,0.9622494,0.007646262
GNG_region_11_AAACCAACACACAATT-1,SeuratProject,AAACCAACACACAATT-1,CTTTATCGTCGGTTTG-1,1,0,15095,14589,986,3928,8756,⋯,18.005190,0.008573655,High,NS < 4,0.195009470,singlet,1.000000e+00,1.000000e+00,0.8049905,0.979612910


In [ ]:
table(atac_multi$SampleID)


 GNG_region_11  P-1694_S-1694  P-1701_S-1701  P-1709_S-1709  P-1764_S-1766 
          7216           5079          12982          19181           8693 
 P-1779_S-1781  P-1780_S-1780  P-6253_S-8498  P-6337_S-8821  P-6640_S-9581 
         10523           8518           6892           4365           6849 
P-6774_S-10146 
          9438 

In [ ]:
atac_multi$ID <- dplyr::recode(atac_multi$SampleID, GNG_region_11 = 'GNG')
table(atac_multi$ID)


           GNG  P-1694_S-1694  P-1701_S-1701  P-1709_S-1709  P-1764_S-1766 
          7216           5079          12982          19181           8693 
 P-1779_S-1781  P-1780_S-1780  P-6253_S-8498  P-6337_S-8821  P-6640_S-9581 
         10523           8518           6892           4365           6849 
P-6774_S-10146 
          9438 

In [ ]:
meta_id_cells <- data.frame(Cell = colnames(atac_multi),
           ID = atac_multi$ID,
           check.names = FALSE
          )
meta_id_cells

,Cell,ID
,<chr>,<chr>
GNG_region_11_AAACAGCCAAAGCGGC-1,GNG_region_11_AAACAGCCAAAGCGGC-1,GNG
GNG_region_11_AAACAGCCAGGTTCAC-1,GNG_region_11_AAACAGCCAGGTTCAC-1,GNG
GNG_region_11_AAACATGCAACTGGCT-1,GNG_region_11_AAACATGCAACTGGCT-1,GNG
GNG_region_11_AAACATGCACGAACAG-1,GNG_region_11_AAACATGCACGAACAG-1,GNG
GNG_region_11_AAACATGCAGGCCAAA-1,GNG_region_11_AAACATGCAGGCCAAA-1,GNG
GNG_region_11_AAACCAACACACAATT-1,GNG_region_11_AAACCAACACACAATT-1,GNG
GNG_region_11_AAACCAACACGTTACA-1,GNG_region_11_AAACCAACACGTTACA-1,GNG
GNG_region_11_AAACCAACATCATGTG-1,GNG_region_11_AAACCAACATCATGTG-1,GNG
GNG_region_11_AAACCAACATGATTGT-1,GNG_region_11_AAACCAACATGATTGT-1,GNG


In [ ]:
all(meta_id_cells$ID %in% meta_cohort$ID)

[1] TRUE

In [ ]:
atac_multi_meta <- list(meta_id_cells, meta_cohort) %>% 
                    purrr::reduce(inner_join, by = 'ID') %>% 
                    distinct(Cell, .keep_all = TRUE) %>% 
                    tibble::column_to_rownames('Cell')
atac_multi_meta

,ID,Study,Institute,Preservation_method,Diagnosis,Tumor_type,Tumor_subtype,Location,Source,Clinical_status,⋯,PPM1D,ACVR1,TSHR,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GNG_region_11_AAACAGCCAAAGCGGC-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACAGCCAGGTTCAC-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACATGCAACTGGCT-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACATGCACGAACAG-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACATGCAGGCCAAA-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACCAACACACAATT-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACCAACACGTTACA-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACCAACATCATGTG-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
GNG_region_11_AAACCAACATGATTGT-1,GNG,Ruiz2023,UMCG,Snap-frozen,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A


In [ ]:
atac_multi <- AddMetaData(atac_multi, atac_multi_meta)
atac_multi <- AddMetaData(atac_multi, FetchData(subset(integrated, 
                                                     dataset == 'Multiome'), 
                                              'predicted.id')
                         )
atac_multi$cell_id <- atac_multi$predicted.id
atac_multi$Batch_for_correction <- '10Xv1_nuclei_multiome'
atac_multi$orig.ident <- NULL
atac_multi@meta.data

,gex_barcode,atac_barcode,is_cell,excluded_reason,gex_raw_reads,gex_mapped_reads,gex_conf_intergenic_reads,gex_conf_exonic_reads,gex_conf_intronic_reads,gex_conf_exonic_unique_reads,⋯,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other,predicted.id,cell_id,Batch_for_correction
,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GNG_region_11_AAACAGCCAAAGCGGC-1,AAACAGCCAAAGCGGC-1,ACAGCGGGTGGATTAG-1,0,2,4828,4639,410,1182,2776,1069,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACAGCCAGGTTCAC-1,AAACAGCCAGGTTCAC-1,ACAGCGGGTCCTGACA-1,1,0,11573,10964,1090,1980,5696,1832,⋯,WT,WT,WT,WT,WT,WT,N/A,Differentiated_like_normal,Differentiated_like_normal,10Xv1_nuclei_multiome
GNG_region_11_AAACATGCAACTGGCT-1,AAACATGCAACTGGCT-1,CATTTAGGTCTTGACG-1,1,0,3339,3187,234,648,2149,594,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACATGCACGAACAG-1,AAACATGCACGAACAG-1,CATTTAGGTCACACTG-1,1,0,15407,14885,969,6106,7092,5543,⋯,WT,WT,WT,WT,WT,WT,N/A,Differentiated_like_normal,Differentiated_like_normal,10Xv1_nuclei_multiome
GNG_region_11_AAACATGCAGGCCAAA-1,AAACATGCAGGCCAAA-1,CATTTAGGTTGAGGTA-1,0,2,127096,122482,9153,26593,80542,24065,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACCAACACACAATT-1,AAACCAACACACAATT-1,CTTTATCGTCGGTTTG-1,1,0,15095,14589,986,3928,8756,3453,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACCAACACGTTACA-1,AAACCAACACGTTACA-1,CTTTATCGTTACAGCT-1,0,2,26907,26106,1387,6439,17328,5878,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACCAACATCATGTG-1,AAACCAACATCATGTG-1,CTTTATCGTCCCGGAT-1,0,2,2426,2322,167,554,714,529,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome
GNG_region_11_AAACCAACATGATTGT-1,AAACCAACATGATTGT-1,CTTTATCGTAGGTTAG-1,0,2,7023,6821,320,2345,3762,2105,⋯,WT,WT,WT,WT,WT,WT,N/A,Stem_like_tumor,Stem_like_tumor,10Xv1_nuclei_multiome


### Subset for testing

#### RNA

In [ ]:
# only include 10x generated data
rna_only <- subset(rna_only, Study %in% c('Filbin2018', 'Liu2022'), invert = TRUE)

# Subset
n_cells <- nrow(rna_only@meta.data)
n_sample <- round(n_cells * 0.1)
sampled_cells <- sample(Cells(rna_only), n_sample)
rna_only_subset <- subset(rna_only, cells = sampled_cells)
rna_only_subset

An object of class Seurat 
38576 features across 31118 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
sceasy::convertFormat(rna_only_subset, from="seurat", to="anndata",
                      assay = "RNA", 
                      drop_single_values = FALSE,
                      main_layer = "counts",
                      outFile='data/subset_rna_only_dmg_atlas.h5ad')

saveRDS(rna_only_subset, 'data/subset_rna_only_dmg_atlas.rds')

Warning message:
“`invoke()` is deprecated as of rlang 0.4.0.
Please use `exec()` or `inject()` instead.
This warning is displayed once every 8 hours.”


AnnData object with n_obs × n_vars = 31118 × 19248
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_RAW', 'nFeature_RAW', 'DF.class', 'DF.score', 'scDblFinder.class', 'scDblFinder.score', 'ID', 'SampleID', 'Data', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'nFeature_Diff', 'nCount_Diff', 'Batch_for_correction', 'scDblFinder.clusters.class', 'scDblFinder.clusters.score', 'scDblFinder.class.clusters', 'doublet.combn.fisher', 'doublet.combn.mean', 'doublet.combn.fisher.class', 'doublet.combn.mean.class', 'Study', 'Original_annotation', 'Isolation_method_by_cell', 'Cell_type_granular_mouse_correlations', 'Cell_type_mouse_correlations', 'Cell_type_consensus_Jessa2022', 'Malignant_normal_consensus_Jessa2022', 'Institute', 'Preservation_method', 'Diagnosis', 'Tumor_type', 'Tumor_subtype', 'Location', 'Source', 'Clinical_status', 'Isolation_method', 'Sc_platform_RNA', 'Sc_platform_ATAC', 'Sc_multiome', 'Raw_data_available', 'Counts', 'Genome_version', 'Paired_sample

#### ATAC

In [ ]:
# Subset
n_cells <- nrow(atac_only@meta.data)
n_sample <- round(n_cells * 0.5)
sampled_cells <- sample(Cells(atac_only), n_sample)
atac_only_subset <- subset(atac_only, cells = sampled_cells)
atac_only_subset

An object of class Seurat 
378298 features across 21804 samples within 1 assay 
Active assay: ATAC (378298 features, 0 variable features)
 2 layers present: counts, data

In [ ]:
saveRDS(atac_only_subset, 'data/subset_atac_only_dmg_atlas.rds')

In [ ]:
# change nomenclature of the features to match other down stream tools in python
rownames(atac_only_subset@assays$ATAC@counts) <- sub("-", ":", 
                                                     rownames(atac_only_subset@assays$ATAC@counts))
rownames(atac_only_subset@assays$ATAC@data) <- sub("-", ":", 
                                                   rownames(atac_only_subset@assays$ATAC@data))
rownames(atac_only_subset@assays$ATAC@meta.features) <- sub("-", ":", 
                                                     rownames(atac_only_subset@assays$ATAC@meta.features))

In [ ]:
sceasy::convertFormat(atac_only_subset, from="seurat", to="anndata",
                      assay = "ATAC", 
                      drop_single_values = FALSE,
                      main_layer = "counts",
                      outFile='data/subset_atac_only_dmg_atlas.h5ad')

AnnData object with n_obs × n_vars = 21804 × 378298
    obs: 'total', 'duplicate', 'chimeric', 'unmapped', 'lowmapq', 'mitochondrial', 'nonprimary', 'passed_filters', 'is__cell_barcode', 'excluded_reason', 'TSS_fragments', 'DNase_sensitive_region_fragments', 'enhancer_region_fragments', 'promoter_region_fragments', 'on_target_fragments', 'blacklist_region_fragments', 'peak_region_fragments', 'peak_region_cutsites', 'nCount_ATAC', 'nFeature_ATAC', 'SampleID', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'pct_reads_in_peaks', 'blacklist_ratio', 'high.tss', 'nucleosome_group', 'scDblFinder.score', 'scDblFinder.class', 'amulet_pvalue', 'amulet_qvalue', 'scDblFinder.p', 'dbl_combined_score', 'ID', 'Study', 'Institute', 'Preservation_method', 'Diagnosis', 'Tumor_type', 'Tumor_subtype', 'Location', 'Source', 'Clinical_status', 'Isolation_method', 'Sc_platform_RNA', 'Sc_platform_ATAC', 'Sc_multiome', 'Raw_data_available', 'Counts', 'Genome_version', 'Paired

#### Multiome

Sicne some cells are only present in one of the two assays, we will initiallty only test on those that have paired information

In [ ]:
# Get the cell names present in obj1 but not in obj2
cells_only_in_rna <- setdiff(rownames(rna_multi@meta.data), rownames(atac_multi@meta.data))

# Get the cell names present in obj2 but not in obj1
cells_only_in_atac <- setdiff(rownames(atac_multi@meta.data), rownames(rna_multi@meta.data))

# Print the results
print(paste("Cell names present in rna but not in atac:", length(cells_only_in_rna)))
# print(cells_only_in_rna)

print(paste("Cell names present in atac but not in rna:", length(cells_only_in_atac)))
# print(cells_only_in_atac)


[1] "Cell names present in rna but not in atac: 27028"
[1] "Cell names present in atac but not in rna: 40152"


In [ ]:
common_cells <- intersect(rownames(rna_multi@meta.data), rownames(atac_multi@meta.data))

rna_multi <- subset(rna_multi, cells = common_cells)
atac_multi <- subset(atac_multi, cells = common_cells)
rna_multi
atac_multi
gc()

An object of class Seurat 
38576 features across 59584 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

An object of class Seurat 
378298 features across 59584 samples within 1 assay 
Active assay: ATAC (378298 features, 0 variable features)
 2 layers present: counts, data

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,11415703,609.7,16520452,882.3,16520452,882.3
Vcells,21785007777,166206.5,36144084056,275757.5,30069779978,229414.3


In [ ]:
# Subset
n_cells <- nrow(rna_multi@meta.data)
n_sample <- round(n_cells * 0.3)
sampled_cells <- sample(Cells(rna_multi), n_sample)
rna_multi_subset <- subset(rna_multi, cells = sampled_cells)
rna_multi_subset
atac_multi_subset <- subset(atac_multi, cells = sampled_cells)
atac_multi_subset

An object of class Seurat 
38576 features across 17875 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

An object of class Seurat 
378298 features across 17875 samples within 1 assay 
Active assay: ATAC (378298 features, 0 variable features)
 2 layers present: counts, data

In [ ]:
sceasy::convertFormat(rna_multi_subset, from="seurat", to="anndata",
                      assay = "RNA", 
                      drop_single_values = FALSE,
                      main_layer = "counts",
                      outFile='data/subset_rna_multi_dmg_atlas.h5ad')

saveRDS(rna_multi_subset, 'data/subset_rna_multi_dmg_atlas.rds')

AnnData object with n_obs × n_vars = 17875 × 19248
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_RAW', 'nFeature_RAW', 'DF.class', 'DF.score', 'scDblFinder.class', 'scDblFinder.score', 'ID', 'SampleID', 'Data', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'nFeature_Diff', 'nCount_Diff', 'Batch_for_correction', 'scDblFinder.clusters.class', 'scDblFinder.clusters.score', 'scDblFinder.class.clusters', 'doublet.combn.fisher', 'doublet.combn.mean', 'doublet.combn.fisher.class', 'doublet.combn.mean.class', 'Study', 'Original_annotation', 'Isolation_method_by_cell', 'Cell_type_granular_mouse_correlations', 'Cell_type_mouse_correlations', 'Cell_type_consensus_Jessa2022', 'Malignant_normal_consensus_Jessa2022', 'Institute', 'Preservation_method', 'Diagnosis', 'Tumor_type', 'Tumor_subtype', 'Location', 'Source', 'Clinical_status', 'Isolation_method', 'Sc_platform_RNA', 'Sc_platform_ATAC', 'Sc_multiome', 'Raw_data_available', 'Counts', 'Genome_version', 'Paired_sample

In [ ]:
saveRDS(atac_multi_subset, 'data/subset_atac_multi_dmg_atlas.rds')

In [ ]:
# change nomenclature of the features to match other down stream tools in python
rownames(atac_multi_subset@assays$ATAC@counts) <- sub("-", ":", 
                                                     rownames(atac_multi_subset@assays$ATAC@counts))
rownames(atac_multi_subset@assays$ATAC@data) <- sub("-", ":", 
                                                   rownames(atac_multi_subset@assays$ATAC@data))
rownames(atac_multi_subset@assays$ATAC@meta.features) <- sub("-", ":", 
                                                   rownames(atac_multi_subset@assays$ATAC@meta.features))

In [ ]:
# to avoid later copnflits with cell_id coming from RAN and ATAC we chnage the ATAC predictions
# to the labels assigned in the RNA counterpart

atac_multi_subset <- AddMetaData(atac_multi_subset, FetchData(rna_multi_subset, 'cell_id'))

sceasy::convertFormat(atac_multi_subset, from="seurat", to="anndata",
                      assay = "ATAC", 
                      drop_single_values = FALSE,
                      main_layer = "counts",
                      outFile='data/subset_atac_multi_dmg_atlas.h5ad')

AnnData object with n_obs × n_vars = 17875 × 378298
    obs: 'gex_barcode', 'atac_barcode', 'is_cell', 'excluded_reason', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'passed_filters', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'nCount_ATAC', 'nFeature_ATAC', 'SampleID', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'pct_reads_in_peaks', 'blacklist_ratio', 'high.tss', 'nucleosome_group', 'scDblFinder.score', 'scDblFi

In [ ]:
sessionInfo()

R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.7 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.32.0           sceasy_0.0.7            reticulate_1.28        
[4] dplyr_1.1.1             Seurat_4.9.9.9041       SeuratObject_4.9.9.9083
[7] sp_1.6-0                Signac_1.9.0.9000      

loaded via a namespace (and not attached):
  [1] readxl_1.4.2                